In [0]:
%sql
SELECT `_updated_at` FROM la_lakehouse.bronze.la_building_permits_issued LIMIT 5;

In [0]:
INTERNAL_CONFIG = [
    {
        "API_URL": "https://data.lacity.org/resource/qyra-qm2s.csv",
        "DELTA_TABLE_NAME": "la_lakehouse.bronze.la_parcels",
        "PARMAS": {
           "$limit" : 10000,
            "$offset" : 0,
            "$order" : "id",
            "$where": None,
            "$select": "*,:updated_at" 
        }
    },
    {
        "API_URL": "https://data.lacity.org/resource/pi9x-tg5x.csv",
        "DELTA_TABLE_NAME": "la_lakehouse.bronze.la_building_permits_issued",
        "PARMAS": {
           "$limit" : 10000,
            "$offset" : 0,
            "$order" : "permit_nbr",
            "$where": None,
            "$select": "*,:updated_at" 
        }
    }
]

In [0]:
import pandas as pd
import requests
import io
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
APP_TOKEN = dbutils.secrets.get(scope="la-lakehouse", key="socrata-app-token")
headers = {
        "X-App-Token": APP_TOKEN
    }

for config in INTERNAL_CONFIG:
    API_URL = config["API_URL"]
    DELTA_TABLE_NAME = config["DELTA_TABLE_NAME"]
    print(f"Loading {DELTA_TABLE_NAME} from {API_URL}")

    offset = config["PARMAS"]["$offset"]
    limit = config["PARMAS"]["$limit"]
    
    # where parameter
    max_date = spark.sql(f"SELECT MAX(_updated_at) FROM {DELTA_TABLE_NAME}").collect()[0][0]

    soql_filter = f":updated_at > '{max_date}'"
    config['PARMAS']['$where'] = soql_filter

    is_first_chunk = True
    table_schema = None

    while True:

        response = requests.get(API_URL, headers=headers, params=config['PARMAS'])

        if response.status_code != 200:
            print(f"API Error! {response.status_code}: {response.text}")
            break

        df_chunk = pd.read_csv(io.StringIO(response.text), low_memory=False, dtype=str)
        
        if df_chunk.empty:
            break
        df_chunk = df_chunk.rename(columns={":updated_at": "_updated_at"})
        if table_schema is None:
            table_schema = StructType([StructField(col, StringType(), True) for col in df_chunk.columns])

        spark_chunk_df = spark.createDataFrame(df_chunk)

        STAGING_TABLE = f"{DELTA_TABLE_NAME}_staging"

        print(f"Rows in this chunk: {len(df_chunk)}")
        
        if is_first_chunk:
            spark_chunk_df.write.mode("overwrite").format('delta').saveAsTable(STAGING_TABLE) 
            is_first_chunk = False
        else:
            spark_chunk_df.write.mode("append").format('delta').saveAsTable(STAGING_TABLE)


        offset += limit
        config['PARMAS']['$offset'] = offset

        if len(df_chunk) < limit:
            break

    if not is_first_chunk: 
        print(f"\nMerging new data into {DELTA_TABLE_NAME} ...")  
        spark.sql(f"""
                  MERGE INTO {DELTA_TABLE_NAME} AS T 
                  USING (SELECT * FROM {STAGING_TABLE}) AS S 
                  ON T.{config["PARMAS"]["$order"]} = S.{config["PARMAS"]["$order"]}
                   WHEN MATCHED THEN UPDATE SET *
                   WHEN NOT MATCHED THEN INSERT *
                  """)
        spark.sql(f"DROP TABLE IF EXISTS {STAGING_TABLE}")

    print(f"Finished processing and fully updated: {DELTA_TABLE_NAME}\n")    
